Gather updated lists of files, samples, metadata\
KL 2 April 2026\
Set up a database, just with sample information

In [141]:
%reset -f
#%whos #also useful at times

In [142]:
import pandas as pd
import os
import pdb

#need this to see the full column width
pd.set_option('display.max_colwidth', None)

In [147]:
# #now I see why Ben was deleting the database...otherwise get multiple inserts
# but I cannot get this to work as it is still in use and I am having trouble closing it.
# %tried;
# session.close()
# engine.dispose()
# def delete_db():
#     print('Deleting database')
#     db_path = 'new_database.db'
#     if os.path.exists(db_path):
# #         os.unlink(db_path)
#         os.remove(db_path)

In [148]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [149]:
# define the classes

class DiscreteInfo(Base):
    __tablename__ = 'discrete'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cruise = Column(String)
    cast = Column(String)
    niskin = Column(String)
    nominalDepth = Column(String)
    
    #latest, do I need the repr(self?)
    
class SeqInfo(Base):
    __tablename__ = 'sequencing'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    getDataHere = Column(String) #this will be the name of the square data file...
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, fullname={self.filename!r})"
    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"index(id={self.id!r}, filename={self.filename!r})"

In [150]:
# create the database tables
Base.metadata.create_all(engine)

In [151]:
# # insert some data, setup functions, one per data type
def load_sequencing_info():
    print('Loading sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = SeqInfo()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FilenameinCyverse']
        db.getDataHere = fName
        session.add(db)
    
    session.commit()
    

def load_cyverse_info():
    print('Loading sequencing information')
    dataDir = '../test_data/BIOS-SCOPE time series/'
    fName = 'files_shortList.txt'
    df = pd.read_csv(os.path.join(dataDir,fName),sep='\t',header=None,comment = '#')

    #strip off the end of the filename
    for index,row in df.iterrows():
        #file = os.path.basename(row.to_string()).strip('fastq.gz')
        file = os.path.basename(row.to_string()).strip('.gz')
        df.loc[index,'filename'] = file

    session = Session()
    for index, row in df.iterrows():
        db = CyverseInfo()
        db.filename = row['filename'] 
        session.add(db)
    
    session.commit()

def load_discrete_info():
    print('Loading discrete sample information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'BATS_BS_COMBINED_MASTER_mini.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name='DATA'))

    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = DiscreteInfo()
        db.bottleID = row['New_ID'] 
        db.cruise = row['Cruise_ID']
        db.cast = row['Cast']
        db.niskin = row['Niskin']
        db.nominalDepth = row['Nominal_Depth']
        session.add(db)
    
    session.commit()

In [152]:
#now run the functions
load_sequencing_info()
load_cyverse_info()
load_discrete_info()

Loading sequencing information
Loading sequencing information
Loading discrete sample information


In [158]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'sequencing']


In [159]:
from sqlalchemy import create_engine, inspect, MetaData, Table

user_seq = Table('sequencing', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)

user_discrete

Table('discrete', MetaData(), Column('id', INTEGER(), table=<discrete>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<discrete>), Column('cruise', VARCHAR(), table=<discrete>), Column('cast', VARCHAR(), table=<discrete>), Column('niskin', VARCHAR(), table=<discrete>), Column('nominalDepth', VARCHAR(), table=<discrete>), schema=None)

In [160]:
user_seq

Table('sequencing', MetaData(), Column('id', INTEGER(), table=<sequencing>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<sequencing>), Column('cast', VARCHAR(), table=<sequencing>), Column('NominalDepth', VARCHAR(), table=<sequencing>), Column('filename', VARCHAR(), table=<sequencing>), Column('getDataHere', VARCHAR(), table=<sequencing>), schema=None)

In [161]:
stmt = select(
    user_seq.c.bottleID, 
    user_cy.c.filename
 ).join_from(
    user_seq,
    user_cy,
    user_cy.c.filename == user_seq.c.filename
)

#session.scalars(stmt).one()
with Session(engine) as session:
    for row in session.execute(stmt):
        print(row)

('1035501701', 'lane1-s001-indexN716-D-S518-D-ACTCGCTA-CTATTAAG-BSv4-342_S1_L001_R1_001.fastq')
('1035501703', 'lane1-s002-indexN716-D-S520-D-ACTCGCTA-AAGGCTAT-BSv4-343_S2_L001_R1_001.fastq')
('1035501705', 'lane1-s003-indexN716-D-S521-D-ACTCGCTA-GAGCCTTA-BSv4-344_S3_L001_R1_001.fastq')
('1035501707', 'lane1-s004-indexN716-D-S522-D-ACTCGCTA-TTATGCGA-BSv4-345_S4_L001_R1_001.fastq')
('1035501709', 'lane1-s005-indexN716-D-S513-D-ACTCGCTA-TCGACTAG-BSv4-346_S5_L001_R1_001.fastq')
('1035501719', 'lane1-s010-indexN718-D-S520-D-GGAGCTAC-AAGGCTAT-BSv4-351_S10_L001_R1_001.fastq')
('1035501721', 'lane1-s011-indexN718-D-S521-D-GGAGCTAC-GAGCCTTA-BSv4-352_S11_L001_R1_001.fastq')
('1035501723', 'lane1-s012-indexN718-D-S522-D-GGAGCTAC-TTATGCGA-BSv4-353_S12_L001_R1_001.fastq')
('1035601501', 'lane1-s013-indexN718-D-S513-D-GGAGCTAC-TCGACTAG-BSv4-354_S13_L001_R1_001.fastq')
('1035601503', 'lane1-s014-indexN718-D-S515-D-GGAGCTAC-TTCTAGCT-BSv4-355_S14_L001_R1_001.fastq')
('1035601505', 'lane1-s015-indexN71

In [163]:
#now, with the discrete data, find the rows there with matching Bottle ID in the seqInfo file

In [174]:
#set this up as a left outer join (all rows of discrete and only those rows of seqdata that match)
#start tidying this up to make it useful. Plan is to ultimately send out one table with all
#the discrete information and columns for cases where there is V1V2, V4, mtabs...

stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    #user_seq.c.bottleID.label('changeThis')
    user_seq.c.getDataHere
).join_from(
    user_discrete,
    user_seq,
    user_discrete.c.bottleID == user_seq.c.bottleID,
    isouter=True
)

#session.scalars(stmt).one()
with Session(engine) as session:
#     for row in session.execute(stmt):
#         print(row)
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)

In [175]:
df.head()

,New_ID,bottleID,cast,cruise,getDataHere,id,niskin,nominalDepth
0,1033900707,1033900707,7,AE1718,None,1,7,40
1,1033900708,1033900708,7,AE1718,None,2,8,40
2,1033900709,1033900709,7,AE1718,None,3,9,60
3,1033900710,1033900710,7,AE1718,None,4,10,60
4,1033900711,1033900711,7,AE1718,None,5,11,80


In [176]:
df.to_csv('temp2.csv',index=False)

In [433]:
session.close()

In [ ]:
#Stick some code below this spot as a holding zone

raise SystemExit("Stop execution here")

In [11]:
from sqlalchemy import Boolean, Column, ForeignKey, Integer, String, Date, Float, DateTime
from sqlalchemy.orm import relationship

#moved the database.py file into the 'notebooks' folder, so this next line reads database.py
#from database import Base



In [12]:
from sqlalchemy import create_engine

SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"

engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)

In [13]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [14]:
from sqlalchemy import Table, Column, Integer, String
user_table_seqInfo = Table(
    "seqInfo",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("bottleID", String(30)),
    Column("cast", String),
    Column("NominalDepth", String),
    Column("filename", String),
)

In [15]:
user_table_seqInfo.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<seqInfo>, primary_key=True, nullable=False))

In [16]:
metadata_obj.create_all(engine)

2026-04-03 10:19:09,962 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 10:19:09,964 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("seqInfo")
2026-04-03 10:19:09,965 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:09,968 INFO sqlalchemy.engine.Engine COMMIT


In [17]:
#note sure I udnerstand this, but enter for now
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [18]:
Base.metadata

MetaData()

In [19]:
##now we want to declare our classes
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

#mote difference in syntax from example...this is new in SQLAlchemy 1.4

class SeqInfo(Base):
    __tablename__ = 'sequencingInfo'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, fullname={self.filename!r})"
    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.filename!r})"

In [20]:
Base.metadata.create_all(engine)

2026-04-03 10:19:10,038 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 10:19:10,039 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("sequencingInfo")
2026-04-03 10:19:10,041 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:10,043 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("cyverse")
2026-04-03 10:19:10,045 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:10,046 INFO sqlalchemy.engine.Engine COMMIT


In [21]:
metadata_obj

MetaData()

In [22]:
user_table_seqInfo = Table("sequencingInfo", metadata_obj, autoload_with=engine)

2026-04-03 10:19:10,085 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 10:19:10,087 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("sequencingInfo")
2026-04-03 10:19:10,087 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:10,089 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2026-04-03 10:19:10,090 INFO sqlalchemy.engine.Engine [raw sql] ('sequencingInfo',)
2026-04-03 10:19:10,092 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("sequencingInfo")
2026-04-03 10:19:10,093 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:10,096 INFO sqlalchemy.engine.Engine PRAGMA temp.foreign_key_list("sequencingInfo")
2026-04-03 10:19:10,097 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-03 10:19:10,098 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE nam

In [23]:
#now insert data

In [24]:
from sqlalchemy.orm import sessionmaker, Session
from datetime import datetime, time
from tqdm import tqdm

In [25]:
Session = sessionmaker()

Session.configure(bind=engine)

In [32]:
def load_sequencing_info():
    print('Loading sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db_si = SeqInfo()
        db_si.bottleID = row['BottleID'] 
        db_si.cast = row['Cast']
        #db_si.filename = row['FilenameInCyverse']
        session.add(db_si)
    session.commit()

In [33]:
load_sequencing_info()

Loading sequencing information
2026-04-03 10:19:40,771 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-03 10:19:40,821 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,822 INFO sqlalchemy.engine.Engine [generated in 0.00827s (insertmanyvalues) 1/2304 (ordered; batch not supported)] (1035501701, '017', None, None)
2026-04-03 10:19:40,826 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,827 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2304 (ordered; batch not supported)] (1035501703, '017', None, None)
2026-04-03 10:19:40,828 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,829 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/2304 (ordered; batch not 

2026-04-03 10:19:40,876 INFO sqlalchemy.engine.Engine [insertmanyvalues 27/2304 (ordered; batch not supported)] (1035700705, '007', None, None)
2026-04-03 10:19:40,876 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,878 INFO sqlalchemy.engine.Engine [insertmanyvalues 28/2304 (ordered; batch not supported)] (1035700707, '007', None, None)
2026-04-03 10:19:40,878 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,879 INFO sqlalchemy.engine.Engine [insertmanyvalues 29/2304 (ordered; batch not supported)] (1035700709, '007', None, None)
2026-04-03 10:19:40,880 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,880 INFO sqlalchemy.engine.Engine [insertmanyvalues 30/2304

2026-04-03 10:19:40,921 INFO sqlalchemy.engine.Engine [insertmanyvalues 54/2304 (ordered; batch not supported)] ('2035800211', '002', None, None)
2026-04-03 10:19:40,923 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,923 INFO sqlalchemy.engine.Engine [insertmanyvalues 55/2304 (ordered; batch not supported)] ('2035800213', '002', None, None)
2026-04-03 10:19:40,924 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,926 INFO sqlalchemy.engine.Engine [insertmanyvalues 56/2304 (ordered; batch not supported)] ('2035800215', '002', None, None)
2026-04-03 10:19:40,926 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,927 INFO sqlalchemy.engine.Engine [insertmanyvalues 5

2026-04-03 10:19:40,966 INFO sqlalchemy.engine.Engine [insertmanyvalues 81/2304 (ordered; batch not supported)] (9191000401, '004', None, None)
2026-04-03 10:19:40,967 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,967 INFO sqlalchemy.engine.Engine [insertmanyvalues 82/2304 (ordered; batch not supported)] (9191000403, '004', None, None)
2026-04-03 10:19:40,969 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,971 INFO sqlalchemy.engine.Engine [insertmanyvalues 83/2304 (ordered; batch not supported)] (9191000405, '004', None, None)
2026-04-03 10:19:40,971 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:40,972 INFO sqlalchemy.engine.Engine [insertmanyvalues 84/2304

2026-04-03 10:19:41,011 INFO sqlalchemy.engine.Engine [insertmanyvalues 108/2304 (ordered; batch not supported)] (1036001907, '019', None, None)
2026-04-03 10:19:41,011 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:41,013 INFO sqlalchemy.engine.Engine [insertmanyvalues 109/2304 (ordered; batch not supported)] (1036001909, '019', None, None)
2026-04-03 10:19:41,014 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:41,014 INFO sqlalchemy.engine.Engine [insertmanyvalues 110/2304 (ordered; batch not supported)] (1036001911, '019', None, None)
2026-04-03 10:19:41,015 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:41,016 INFO sqlalchemy.engine.Engine [insertmanyvalues 111/

...

2026-04-03 10:19:44,518 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,518 INFO sqlalchemy.engine.Engine [insertmanyvalues 2191/2304 (ordered; batch not supported)] ('9221300701', '007', None, None)
2026-04-03 10:19:44,519 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,520 INFO sqlalchemy.engine.Engine [insertmanyvalues 2192/2304 (ordered; batch not supported)] ('9221300705', '007', None, None)
2026-04-03 10:19:44,521 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,522 INFO sqlalchemy.engine.Engine [insertmanyvalues 2193/2304 (ordered; batch not supported)] ('9221300708', '007', None, None)
2026-04-03 10:19:44,523 INFO sqlalchemy.engine.Engine INSERT INTO "

2026-04-03 10:19:44,565 INFO sqlalchemy.engine.Engine [insertmanyvalues 2217/2304 (ordered; batch not supported)] ('9221301221', '012', None, None)
2026-04-03 10:19:44,566 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,567 INFO sqlalchemy.engine.Engine [insertmanyvalues 2218/2304 (ordered; batch not supported)] ('9221300101', '001', None, None)
2026-04-03 10:19:44,568 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,569 INFO sqlalchemy.engine.Engine [insertmanyvalues 2219/2304 (ordered; batch not supported)] ('9221300106', '001', None, None)
2026-04-03 10:19:44,570 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,571 INFO sqlalchemy.engine.Engine [insertmanyva

2026-04-03 10:19:44,611 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,611 INFO sqlalchemy.engine.Engine [insertmanyvalues 2244/2304 (ordered; batch not supported)] ('9231500605', '006', None, None)
2026-04-03 10:19:44,612 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,613 INFO sqlalchemy.engine.Engine [insertmanyvalues 2245/2304 (ordered; batch not supported)] ('9231500608', '006', None, None)
2026-04-03 10:19:44,615 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,616 INFO sqlalchemy.engine.Engine [insertmanyvalues 2246/2304 (ordered; batch not supported)] ('9231500610', '006', None, None)
2026-04-03 10:19:44,617 INFO sqlalchemy.engine.Engine INSERT INTO "

2026-04-03 10:19:44,660 INFO sqlalchemy.engine.Engine [insertmanyvalues 2270/2304 (ordered; batch not supported)] ('9231501516', '015', None, None)
2026-04-03 10:19:44,661 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,661 INFO sqlalchemy.engine.Engine [insertmanyvalues 2271/2304 (ordered; batch not supported)] ('9231501518', '015', None, None)
2026-04-03 10:19:44,663 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,663 INFO sqlalchemy.engine.Engine [insertmanyvalues 2272/2304 (ordered; batch not supported)] ('9231501520', '015', None, None)
2026-04-03 10:19:44,664 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,665 INFO sqlalchemy.engine.Engine [insertmanyva

2026-04-03 10:19:44,705 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,706 INFO sqlalchemy.engine.Engine [insertmanyvalues 2297/2304 (ordered; batch not supported)] (nan, nan, None, None)
2026-04-03 10:19:44,707 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,707 INFO sqlalchemy.engine.Engine [insertmanyvalues 2298/2304 (ordered; batch not supported)] (nan, nan, None, None)
2026-04-03 10:19:44,708 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cast", "NominalDepth", filename) VALUES (?, ?, ?, ?) RETURNING id
2026-04-03 10:19:44,709 INFO sqlalchemy.engine.Engine [insertmanyvalues 2299/2304 (ordered; batch not supported)] (nan, nan, None, None)
2026-04-03 10:19:44,710 INFO sqlalchemy.engine.Engine INSERT INTO "sequencingInfo" ("bottleID", "cas

In [35]:
Session


sessionmaker(class_='Session', bind=Engine(sqlite:///new_database.db), autoflush=True, expire_on_commit=True)

In [38]:
from sqlalchemy import inspect

session = inspect(Session).session

NoInspectionAvailable: No inspection system is available for object of type <class 'sqlalchemy.orm.session.sessionmaker'>

In [36]:
type(Session)

sqlalchemy.orm.session.sessionmaker

In [ ]:
data_dir = '../test_data/BIOS-SCOPE time series/'

#careful the csv file and the xlsx have the same name but different information, I need the xlsx file
# fName = 'V4_dada2_read_info_03052026.csv'
#df_info = pd.DataFrame(pd.read_csv(os.path.join(data_dir,fName)))
fName = 'V4_dada2_read_info_03052026.xlsx'
df_info = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))

In [ ]:
#start with the sequence data, Luis gave me three lists. 
#Merge these and pull out relevant details, export to CSV file that will be read into the database
seqDir = '../test_data/Luis_fileLists'

dir_list = os.listdir(seqDir)
dir_list_full = [os.path.join(seqDir,f) for f in os.listdir(seqDir)] 